# 🦀 Day 2: Closures as Parameters & Return Values

---

In [ ]:
// Closures as function parameters
fn apply<T, F: Fn(T) -> T>(value: T, f: F) -> T {
    f(value)
}

let result = apply(5, |x| x * x);
println!("5² = {}", result);

let result = apply(String::from("hello"), |s| s.to_uppercase());
println!("Upper: {}", result);

In [ ]:
// Building a configurable sorter
fn sort_by_key<T, K: Ord, F: Fn(&T) -> K>(items: &mut Vec<T>, key_fn: F) {
    items.sort_by(|a, b| key_fn(a).cmp(&key_fn(b)));
}

#[derive(Debug)]
struct Student { name: String, grade: u32 }

let mut students = vec![
    Student { name: "Charlie".into(), grade: 85 },
    Student { name: "Alice".into(), grade: 92 },
    Student { name: "Bob".into(), grade: 88 },
];

sort_by_key(&mut students, |s| s.grade);
println!("By grade: {:#?}", students);

sort_by_key(&mut students, |s| s.name.clone());
println!("By name: {:#?}", students);

In [ ]:
// Returning closures
fn make_adder(n: i32) -> impl Fn(i32) -> i32 {
    move |x| x + n
}

fn make_multiplier(n: i32) -> impl Fn(i32) -> i32 {
    move |x| x * n
}

fn make_clamper(min: i32, max: i32) -> impl Fn(i32) -> i32 {
    move |x| x.max(min).min(max)
}

let add10 = make_adder(10);
let mul5 = make_multiplier(5);
let clamp = make_clamper(0, 100);

println!("add10(32) = {}", add10(32));
println!("mul5(7) = {}", mul5(7));
println!("clamp(150) = {}", clamp(150));
println!("clamp(-5) = {}", clamp(-5));

In [ ]:
// Closure composition — apply functions in sequence
fn compose<F, G>(f: F, g: G) -> impl Fn(i32) -> i32
where
    F: Fn(i32) -> i32,
    G: Fn(i32) -> i32,
{
    move |x| g(f(x))
}

let add5_then_double = compose(make_adder(5), make_multiplier(2));
println!("(10 + 5) × 2 = {}", add5_then_double(10));

let double_then_add5 = compose(make_multiplier(2), make_adder(5));
println!("(10 × 2) + 5 = {}", double_then_add5(10));

In [ ]:
// Dynamic dispatch with Box<dyn Fn>
fn make_operation(op: &str) -> Box<dyn Fn(f64, f64) -> f64> {
    match op {
        "add" => Box::new(|a, b| a + b),
        "sub" => Box::new(|a, b| a - b),
        "mul" => Box::new(|a, b| a * b),
        "div" => Box::new(|a, b| if b != 0.0 { a / b } else { f64::NAN }),
        _ => Box::new(|_, _| 0.0),
    }
}

for op_name in ["add", "sub", "mul", "div"] {
    let op = make_operation(op_name);
    println!("{}: 10 op 3 = {:.2}", op_name, op(10.0, 3.0));
}

In [ ]:
// Practical: event handler registry
use std::collections::HashMap;

type Handler = Box<dyn Fn(&str)>;

struct EventEmitter {
    handlers: HashMap<String, Vec<Handler>>,
}

impl EventEmitter {
    fn new() -> Self { EventEmitter { handlers: HashMap::new() } }

    fn on(&mut self, event: &str, handler: impl Fn(&str) + 'static) {
        self.handlers.entry(event.to_string())
            .or_default()
            .push(Box::new(handler));
    }

    fn emit(&self, event: &str, data: &str) {
        if let Some(handlers) = self.handlers.get(event) {
            for handler in handlers {
                handler(data);
            }
        }
    }
}

let mut emitter = EventEmitter::new();
emitter.on("click", |data| println!("  Handler 1: clicked {}", data));
emitter.on("click", |data| println!("  Handler 2: logging {}", data));
emitter.on("hover", |data| println!("  Hovering over {}", data));

println!("Emitting 'click':");
emitter.emit("click", "button");
println!("Emitting 'hover':");
emitter.emit("hover", "menu");

## 🏋️ Exercises

In [ ]:
// Exercise 1: Write fn apply_pipeline(value: i32, steps: &[Box<dyn Fn(i32) -> i32>]) -> i32
// that applies each function in sequence

// YOUR CODE HERE


## 🎯 Key Takeaways

1. `impl Fn(T) -> U` for static dispatch (one concrete type)
2. `Box<dyn Fn(T) -> U>` for dynamic dispatch (multiple types)
3. `move` closures own their data — essential for returning closures
4. Compose closures to build complex behavior from simple parts
5. Event systems use `Box<dyn Fn>` for flexible handler storage

---
**Tomorrow:** Iterator trait! 🔁